In [ ]:
from IPython.display import display, HTML

display(HTML("""
<div style="background: linear-gradient(135deg, #003049 0%, #023e8a 40%, #0077b6 70%, #00b4d8 100%);
            padding: 50px 40px; border-radius: 12px; text-align: center;
            font-family: Georgia, serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3);">

  <span style="color: #90e0ef; font-weight: bold; font-size: 14px; letter-spacing: 2px;">
    CUC - COLEGIO UNIVERSITARIO DE CARTAGO
  </span>

  <hr style="border: 1px solid rgba(144,224,239,0.5); margin: 15px 0;">

  <h1 style="color: #caf0f8; font-size: 36px; margin: 20px 0 10px 0;
             text-shadow: 2px 2px 4px rgba(0,0,0,0.5); letter-spacing: 2px;">
    Prediccion del Consumo de Energia<br>en Costa Rica
  </h1>

  <h2 style="color: white; font-size: 20px; font-weight: normal; margin: 5px 0 25px 0;">
    Analisis Exploratorio de Datos (EDA)
  </h2>

  <hr style="border: 1px solid rgba(144,224,239,0.5); margin: 15px 0;">

  <p style="color: #90e0ef; font-weight: bold; font-size: 15px; margin: 8px 0;">
    Curso: <span style="color: white; font-weight: normal;">BD-143 Programacion II</span></p>
  <p style="color: #90e0ef; font-weight: bold; font-size: 15px; margin: 8px 0;">
    Profesor: <span style="color: white; font-weight: normal;">Osvaldo Gonzalez Chaves</span></p>
  <p style="color: #90e0ef; font-weight: bold; font-size: 15px; margin: 8px 0;">
    Proyecto: <span style="color: white; font-weight: normal;">Proyecto 5</span></p>
  <p style="color: #90e0ef; font-weight: bold; font-size: 15px; margin: 8px 0;">
    Cuatrimestre: <span style="color: white; font-weight: normal;">I Cuatrimestre 2026</span></p>

  <hr style="border: 1px solid rgba(144,224,239,0.5); margin: 20px 0 15px 0;">

  <p style="color: #90e0ef; font-size: 13px; letter-spacing: 2px; margin-bottom: 10px;">ELABORADO POR</p>
  <p style="color: white; font-size: 20px; font-weight: bold; margin: 5px 0;">Desmond, Luis Diego y Nadin</p>

</div>
"""))

In [ ]:
from IPython.display import display, HTML

display(HTML("""
<style>
.jp-RenderedMarkdown h1 {
  background: linear-gradient(135deg, #003049, #0077b6, #00b4d8);
  -webkit-background-clip: text;
  -webkit-text-fill-color: transparent;
  background-clip: text;
  font-family: Georgia, serif !important;
  font-size: 2em !important;
  border-bottom: 3px solid #0077b6 !important;
  padding-bottom: 8px !important;
}
.jp-RenderedMarkdown h2 {
  color: #0077b6 !important;
  font-family: Georgia, serif !important;
  border-left: 5px solid #00b4d8 !important;
  padding-left: 12px !important;
}
.jp-RenderedMarkdown h3 {
  color: #023e8a !important;
  border-bottom: 1px solid rgba(0,180,216,0.4) !important;
  padding-bottom: 4px !important;
}
</style>
"""))

# Prediccion del Consumo de Energia en Costa Rica � EDA

Este notebook realiza el analisis exploratorio del dataset final que combina datos de
precios medios de **ARESEP** con variables climaticas de **NASA POWER** por empresa
distribuidora, para el periodo 2020�2025.

El objetivo del EDA es comprender la estructura de los datos, detectar problemas de
calidad (nulos, outliers, valores atipicos) y extraer patrones que sirvan de base
para el modelo supervisado de prediccion de consumo electrico.

## 1. Importaciones y configuracion

Importamos las librerias necesarias para visualizacion y la clase `ProcesadorEDA`
del proyecto, que encapsula toda la logica de carga, limpieza y exploracion de datos.
Tambien configuramos el estilo global de los graficos.

In [ ]:
import sys
sys.path.append('../..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from eda.ProcesadorEDA import ProcesadorEDA
import datos


plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
sns.set_style('whitegrid')

PALETA = ['#003049','#0077b6','#00b4d8','#e9c46a','#e76f51','#2a9d8f','#6a4c93','#f4a261','#264653']

print('Configuracion lista.')

## 2. Carga del dataset

Cargamos el archivo `dataset_final_2020_2025.csv` que fue generado por el pipeline ETL
del proyecto. Este archivo ya integra los datos de ARESEP (ventas, ingresos, tarifas)
con las variables climaticas de NASA POWER (temperatura, viento, nubosidad, precipitacion)
por empresa y mes.

Usamos el metodo `csv_to_df()` de la clase `ProcesadorEDA`, que carga el archivo y
calcula automaticamente metricas basicas del dataset (filas, columnas, nulos, ceros).

In [ ]:
from ProcesadorEDA import ProcesadorEDA

eda = ProcesadorEDA()
params = eda.csv_to_df(
    '../../data/processed/dataset_final_2020_2025.csv',
    chain=False
)

print('=== Resumen del dataset ===')
for k, v in params.items():
    if k != 'Dataframe':
        print(f'  {k}: {v}')

df = eda.df
df.head()

## 3. Estructura y tipos de datos

Antes de analizar, revisamos la estructura del dataset: cuantas filas y columnas tiene,
que tipo de dato tiene cada columna, y cuales son los valores posibles de las variables
categoricas principales (empresa, tarifa, sistema).

Esto nos permite confirmar que los tipos son correctos y entender el alcance del dataset.

In [ ]:
print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
print()

unicos = eda.col_uniques(['Empresa', 'Tarifa', 'Sistema'], chain=False)
for col, (cant, vals) in unicos.items():
    print(f'{col} ({cant} valores unicos):')
    for v in sorted(vals):
        print(f'   - {v}')
    print()

In [ ]:
# Tipos de dato por columna
df.dtypes.to_frame('Tipo').T

## 4. Resumen estadistico descriptivo

El resumen descriptivo muestra las medidas de tendencia central y dispersion
para las variables numericas del dataset. Analizamos las columnas mas relevantes
para el proyecto: ventas, ingresos, precio medio y las variables climaticas principales.

| Medida | Descripcion |
|--------|-------------|
| `count` | Valores no nulos |
| `mean` | Promedio aritmetico |
| `std` | Desviacion estandar |
| `min / max` | Valor minimo y maximo |
| `Q1 / Q3` | Percentiles 25 y 75 |
| `median` | Valor central de la distribucion |

In [ ]:
import pandas as pd

cols_desc = ['Ventas', 'Ingreso con CVG', 'Precio Medio con CVG', 'T2M', 'WS10M', 'RH2M', 'PRECTOTCORR']

resumen = eda.res_descrip(cols_desc, chain=False)

# Agregar mediana al resumen estándar
mediana = df[cols_desc].median().rename('median')
resumen = pd.concat([resumen, mediana.to_frame().T])

resumen.round(2)

### Interpretacion del resumen descriptivo

**Ventas (kWh):**
- La media (28.7M kWh) es muy superior a la mediana (6.7M kWh), lo que indica
  una distribucion asimetrica hacia la derecha: pocas empresas grandes (ICE, CNFL)
  concentran la mayor parte de las ventas.
- La desviacion estandar alta confirma la gran disparidad entre empresas.

**Precio Medio con CVG (�/kWh):**
- Existe variacion considerable entre tarifas: la residencial y alumbrado publico
  tienen precios distintos a la industrial o media tension.

**Variables climaticas:**
- La temperatura promedio (T2M) ronda los 23�C, coherente con el clima tropical de CR.
- La precipitacion (PRECTOTCORR) muestra alta variabilidad entre zonas geograficas.

## 5. Analisis de valores nulos

Identificamos las columnas con valores faltantes usando el metodo `col_nulls()` del
procesador. Ademas, reemplazamos el valor centinela **-999** que usa NASA POWER
para indicar datos no disponibles, ya que de lo contrario distorsionaria los analisis.

Detectar y entender los nulos es clave antes de entrenar cualquier modelo supervisado.

In [ ]:

cols_clima = ['T2M','WS10M','CLOUD_AMT','RH2M','T2M_MAX','T2M_MIN',
              'CLOUD_OD','GWETROOT','TS','PRECTOTCORR',
              'ALLSKY_SFC_SW_DWN','PS','T2MWET','ALLSKY_SFC_SW_DIFF','ALLSKY_SFC_LW_DWN']
df[cols_clima] = df[cols_clima].replace(-999, float('nan'))
eda.df = df

nulos = eda.col_nulls(chain=False)
print('Columnas con valores nulos:')
for col, cant in nulos.items():
    pct = cant / len(df) * 100
    print(f'  {col}: {cant} nulos ({pct:.1f}%)')

### Grafico de nulos

In [ ]:
import matplotlib.pyplot as plt
nulos_pct = pd.Series({col: cant/len(df)*100 for col, cant in nulos.items()})

fig, ax = plt.subplots(figsize=(12, 4))
nulos_pct.sort_values().plot(kind='barh', ax=ax, color='#e76f51', edgecolor='white')
ax.axvline(5, color='gray', linestyle='--', linewidth=1, label='Umbral 5%')
ax.set_title('Porcentaje de valores nulos por columna')
ax.set_xlabel('% Nulos')
ax.legend()
plt.tight_layout()
plt.show()

### Interpretacion

Los nulos en variables climaticas (~1.7%) corresponden exclusivamente a la empresa
**USUARIOS DIRECTOS**, que no tiene coordenadas geograficas asignadas en el cliente
NASA POWER y por eso no recibio datos de clima.

Los nulos en `Abonados` (~27%) son esperados: no todas las tarifas registran
esta variable. Para el modelo de prediccion, esta columna podria excluirse o
imputarse segun la empresa.

## 6. Analisis de outliers

Utilizamos el metodo **IQR (Rango Intercuartilico)** para detectar outliers en las
variables numericas principales. Se considera outlier todo valor por debajo de
`Q1 - 1.5*IQR` o por encima de `Q3 + 1.5*IQR`.

Detectar outliers es importante porque pueden distorsionar los modelos de regresion
si no se tratan adecuadamente antes del entrenamiento.

In [ ]:
cols_outliers = ['Ventas', 'Ingreso con CVG', 'Precio Medio con CVG', 'T2M', 'PRECTOTCORR']
resultados = eda.detect_outliers(cols_outliers, chain=False)

print('=== Deteccion de Outliers (metodo IQR) ===')
for r in resultados:
    pct = r['cantidad'] / len(df) * 100
    print(f"\n  {r['columna']}")
    print(f"    Limite inferior : {r['limite_inf']:,.2f}")
    print(f"    Limite superior : {r['limite_sup']:,.2f}")
    print(f"    Outliers        : {r['cantidad']} ({pct:.1f}%)")

### Boxplots de outliers

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

boxplot_cols = ['Ventas', 'Precio Medio con CVG', 'PRECTOTCORR']
titulos      = ['Ventas (kWh)', 'Precio Medio con CVG (�/kWh)', 'Precipitacion (mm/dia)']
colores      = ['#0077b6', '#e9c46a', '#2a9d8f']

for ax, col, titulo, color in zip(axes, boxplot_cols, titulos, colores):
    ax.boxplot(
        df[col].dropna(),
        patch_artist=True,
        boxprops=dict(facecolor=color, alpha=0.7),
        medianprops=dict(color='black', linewidth=2),
        flierprops=dict(markerfacecolor=color, marker='o', alpha=0.4, markersize=3)
    )
    ax.set_title(titulo)
    ax.set_xticks([])

plt.suptitle('Deteccion de Outliers � Variables principales', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Interpretacion

- **Ventas**: Los outliers superiores corresponden a ICE y CNFL, las dos empresas
  mas grandes del pais. No son errores sino diferencias reales de escala.
- **Precio Medio con CVG**: Los valores extremos se explican por tarifas especiales
  (alumbrado publico, usuarios directos) que tienen estructuras de precio distintas.
- **Precipitacion**: Alta variabilidad entre zonas geograficas de Costa Rica
  (Caribe vs Pacifico), lo cual es esperado y climaticamente correcto.

## 7. Ventas por empresa

Analizamos la distribucion de ventas electricas entre las empresas distribuidoras.
Primero observamos el total acumulado del periodo 2020-2025 para entender la
participacion de cada empresa, y luego la evolucion mensual para detectar tendencias.

### Ventas totales por empresa

In [ ]:
import matplotlib.pyplot as plt

PALETA = plt.cm.tab20.colors

ventas_empresa = df.groupby('Empresa')['Ventas'].sum().sort_values()

fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.barh(
    ventas_empresa.index,
    ventas_empresa.values / 1e9,
    color=PALETA[:len(ventas_empresa)],
    edgecolor='white'
)

# Etiquetas de valor
for bar, val in zip(bars, ventas_empresa.values / 1e9):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}B', va='center', fontsize=9)

ax.set_title('Ventas eléctricas totales acumuladas por empresa (2020–2025)')
ax.set_xlabel('Ventas (miles de millones kWh)')

plt.tight_layout()
plt.show()

### Evolucion mensual de ventas empresas principales

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df_ts = df.groupby(['Año', 'Mes', 'Empresa'])['Ventas'].sum().reset_index()

df_ts['Fecha'] = pd.to_datetime(
    df_ts[['Año', 'Mes']]
    .rename(columns={'Año': 'year', 'Mes': 'month'})
    .assign(day=1)
)

empresas_plot = ['ICE', 'CNFL', 'ESPH', 'COOPELESCA', 'JASEC']

fig, ax = plt.subplots(figsize=(13, 5))

for i, empresa in enumerate(empresas_plot):
    sub = df_ts[df_ts['Empresa'] == empresa]
    ax.plot(
        sub['Fecha'],
        sub['Ventas'] / 1e6,
        label=empresa,
        linewidth=2,
        color=PALETA[i],
        marker='o',
        markersize=2
    )

ax.set_title('Evolución mensual de ventas por empresa (2020–2025)')
ax.set_ylabel('kWh (millones)')
ax.set_xlabel('Fecha')

ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))

plt.tight_layout()
plt.show()

### Interpretacion

- **ICE** domina ampliamente el mercado, con ventas que triplican a las del segundo
  lugar (CNFL). Esto refleja su rol historico como principal operador nacional.
- Se observa una **tendencia creciente** en la mayoria de empresas entre 2020 y 2025,
  lo que coincide con la recuperacion economica postpandemia y el crecimiento del
  parque electrico en Costa Rica.
- Las lineas muestran **estacionalidad leve pero consistente**, con variaciones
  que seran analizadas en la seccion siguiente.

## 8. Estacionalidad mensual

Analizamos si el consumo electrico varia segun el mes del ano.
La estacionalidad es un patron ciclico que se repite cada ano y es una variable
clave para el modelo de prediccion. Se analiza tanto de forma agregada como
desagregada por empresa en un heatmap.

In [ ]:
meses_labels = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Set','Oct','Nov','Dic']
estacional = df.groupby('Mes')['Ventas'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(estacional.index, estacional.values / 1e6,
              color=PALETA[1], edgecolor='white', alpha=0.85)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_labels)
ax.set_title('Ventas promedio por mes � Estacionalidad agregada (todas las empresas)')
ax.set_ylabel('kWh promedio (millones)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}M'))

# Linea de promedio global
promedio = estacional.mean() / 1e6
ax.axhline(promedio, color='#e76f51', linestyle='--', linewidth=1.5,
           label=f'Promedio global: {promedio:.1f}M kWh')
ax.legend()
plt.tight_layout()
plt.show()

### Heatmap: Ventas promedio por Empresa x Mes

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pivot = df.groupby(['Empresa', 'Mes'])['Ventas'].mean().unstack()
pivot.columns = meses_labels

fig, ax = plt.subplots(figsize=(14, 5))

sns.heatmap(
    pivot / 1e6,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Millones kWh'}
)

ax.set_title('Heatmap: Ventas promedio por Empresa y Mes (millones kWh)')
ax.set_ylabel('')

plt.tight_layout()
plt.show()

### Interpretacion

- El grafico de barras muestra una **estacionalidad moderada**: los meses de enero
  a marzo (verano en Costa Rica) tienden a tener mayor consumo, probablemente
  por uso de ventilacion y aires acondicionados en epoca seca.
- El heatmap confirma que **ICE domina todos los meses**, pero tambien revela
  diferencias entre empresas: COOPEGUANACASTE, ubicada en zona seca del Pacifico,
  muestra picos en meses de verano mas marcados que empresas del Valle Central.

## 9. Variables climaticas

Las variables climaticas son las entradas principales del modelo junto con
la informacion de empresa y mes. Analizamos su distribucion, su comportamiento
por zona geografica y su evolucion en el tiempo.

Las 4 variables climaticas mas relevantes para predecir consumo electrico son:
temperatura (`T2M`), viento (`WS10M`), humedad relativa (`RH2M`) y
precipitacion (`PRECTOTCORR`).

### Distribucion de variables climaticas principales


In [ ]:
cols_main   = ['T2M',          'WS10M',               'RH2M',             'PRECTOTCORR']
labels_main = ['Temperatura (�C)', 'Viento (m/s)', 'Humedad relativa (%)', 'Precipitacion (mm/dia)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col, label, color in zip(axes, cols_main, labels_main, PALETA):
    df[col].dropna().plot(kind='hist', bins=30, ax=ax,
                          color=color, edgecolor='white', alpha=0.85)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('')

plt.suptitle('Distribucion de variables climaticas principales (NASA POWER)', y=1.02)
plt.tight_layout()
plt.show()

### Temperatura promedio por empresa en el tiempo

In [ ]:
df_t = df.groupby(['Año', 'Mes', 'Empresa'])['T2M'].mean().reset_index()

df_t['Fecha'] = pd.to_datetime(
    df_t[['Año', 'Mes']]
    .rename(columns={'Año': 'year', 'Mes': 'month'})
    .assign(day=1)
)

fig, ax = plt.subplots(figsize=(13, 5))

for i, empresa in enumerate(empresas_plot):
    sub = df_t[df_t['Empresa'] == empresa]
    ax.plot(
        sub['Fecha'],
        sub['T2M'],
        label=empresa,
        linewidth=1.8,
        color=PALETA[i]
    )

ax.set_title('Temperatura promedio mensual por empresa (T2M °C)')
ax.set_ylabel('°C')
ax.set_xlabel('Fecha')

ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

### Interpretacion

- La **temperatura** varia entre ~19C y ~30C segun la empresa, lo que refleja
  las diferencias altitudinales de Costa Rica: COOPEALFARORUIZ (zona de Alfaro Ruiz,
  alta montana) es consistentemente mas fria que COOPEGUANACASTE (Guanacaste, costa).
- Se observa **estacionalidad climatica clara**: las temperaturas suben en
  los primeros meses del ano (verano) y bajan durante la temporada lluviosa.
- Esta variacion geografica y temporal justifica incluir temperatura y empresa
  como variables de entrada del modelo supervisado.

## 10 Matriz de correlacion: clima vs consumo

La correlacion de Pearson mide la relacion lineal entre dos variables numericas.
Analizamos como se relacionan las variables climaticas con el consumo electrico
(Ventas) para identificar cuales tienen mayor poder predictivo.

| Valor | Interpretacion |
|-------|----------------|
| Cercano a 1 | Correlacion positiva fuerte |
| Cercano a -1 | Correlacion negativa fuerte |
| Cercano a 0 | Sin correlacion lineal |

### Usamos el metodo matrix() del ProcesadorEDA sobre columnas numericas seleccionadas

In [ ]:
cols_corr = ['Ventas','Ingreso con CVG','Precio Medio con CVG',
             'T2M','WS10M','RH2M','CLOUD_AMT','PRECTOTCORR','T2M_MAX']

eda.df = df[cols_corr].dropna()
corr_matrix = eda.matrix(chain=False)
eda.df = df

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, square=True)
ax.set_title('Matriz de correlacion: consumo electrico vs variables climaticas')
plt.tight_layout()
plt.show()

### Correlacion de cada variable climatica con Ventas � barras ordenadas

In [ ]:
corr_ventas = df[cols_clima + ['Ventas']].dropna().corr()['Ventas'].drop('Ventas').sort_values()
colores_bar = ['#e76f51' if v < 0 else '#0077b6' for v in corr_ventas]

fig, ax = plt.subplots(figsize=(12, 5))
corr_ventas.plot(kind='barh', ax=ax, color=colores_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlacion de variables climaticas (NASA POWER) con Ventas kWh')
ax.set_xlabel('Coeficiente de Pearson')
plt.tight_layout()
plt.show()

print('\nTop 3 mayor correlacion positiva con Ventas:')
print(corr_ventas.tail(3).to_string())
print('\nTop 3 mayor correlacion negativa con Ventas:')
print(corr_ventas.head(3).to_string())

### Scatter: Temperatura vs Ventas coloreado por empresa

In [ ]:
df_sc = df[['T2M','Ventas','Empresa']].dropna()

fig, ax = plt.subplots(figsize=(12, 5))
for i, empresa in enumerate(df_sc['Empresa'].unique()):
    sub = df_sc[df_sc['Empresa'] == empresa]
    ax.scatter(sub['T2M'], sub['Ventas'] / 1e6, alpha=0.5,
               s=20, label=empresa, color=PALETA[i % len(PALETA)])

ax.set_title('Temperatura (T2M) vs Ventas � por empresa')
ax.set_xlabel('Temperatura promedio mensual (�C)')
ax.set_ylabel('Ventas (millones kWh)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

### Interpretacion

- La **temperatura (T2M)** y la **temperatura maxima (T2M_MAX)** tienen correlacion
  positiva con las ventas: a mayor calor, mayor consumo electrico. Esto es consistente
  con el uso de ventilacion y aire acondicionado en periodos calidos.
- La **humedad relativa (RH2M)** tiene correlacion negativa: las epocas lluviosas
  (mayor humedad) se asocian con menor consumo, posiblemente por menor necesidad
  de refrigeracion artificial.
- El scatter confirma que la relacion temperatura-ventas **no es uniforme**: ICE
  tiene volumen muy alto independientemente de la temperatura, lo que subraya
  la importancia de incluir la empresa como variable categorica en el modelo.

## 11. Evolucion del ingreso y precio medio

Analizamos como han evolucionado los ingresos totales y el precio medio de la
electricidad a lo largo del periodo. Un aumento sostenido del precio medio
podria indicar ajustes tarifarios de ARESEP, inflacion o cambios en la
estructura de consumo del pais.

In [ ]:
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(14, 5))


ingreso_anual = df.groupby('Año')['Ingreso con CVG'].sum()
ingreso_anual.plot(
    kind='bar',
    ax=axes[0],
    color=PALETA[5],
    edgecolor='white',
    alpha=0.85
)

axes[0].set_title('Ingreso total con CVG por año')
axes[0].set_ylabel('₡ (billones)')
axes[0].set_xlabel('Año')
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e12:.1f}T')
)
axes[0].tick_params(axis='x', rotation=0)

# Precio medio por año – líneas por tarifa
precio_anual = df.groupby(['Año', 'Tarifa'])['Precio Medio con CVG'].mean().reset_index()

tarifas_top = df.groupby('Tarifa')['Ventas'].sum().nlargest(5).index.tolist()

for i, tarifa in enumerate(tarifas_top):
    sub = precio_anual[precio_anual['Tarifa'] == tarifa]
    axes[1].plot(
        sub['Año'],
        sub['Precio Medio con CVG'],
        marker='o',
        label=tarifa,
        color=PALETA[i],
        linewidth=2
    )

axes[1].set_title('Precio medio con CVG por año – tarifas principales')
axes[1].set_ylabel('₡ / kWh')
axes[1].set_xlabel('Año')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### Interpretacion

- El **ingreso total** ha crecido consistentemente de 2020 a 2025, lo que refleja
  tanto el aumento de abonados como los ajustes tarifarios aprobados por ARESEP.
- Todas las tarifas muestran **tendencia alcista en el precio**, con la tarifa
  Industrial experimentando uno de los mayores crecimientos porcentuales.
- Para el modelo supervisado, el ano y el mes son variables temporales importantes
  que capturan esta tendencia ascendente en precios y consumo.

## 12. Resumen estadistico por empresa

Generamos una tabla resumen por empresa que integra las principales metricas
del dataset: volumen de ventas, ingreso generado, precio medio cobrado y
condiciones climaticas promedio de su zona geografica.

In [ ]:
resumen_empresa = df.groupby('Empresa').agg(
    Registros          = ('Ventas', 'count'),
    Ventas_Total_M     = ('Ventas',            lambda x: round(x.sum()  / 1e6, 1)),
    Ventas_Prom_M      = ('Ventas',            lambda x: round(x.mean() / 1e6, 2)),
    Ingreso_Total_B    = ('Ingreso con CVG',   lambda x: round(x.sum()  / 1e9, 1)),
    Precio_Medio       = ('Precio Medio con CVG', 'mean'),
    Temp_Prom_C        = ('T2M',   'mean'),
    Lluvia_Prom_mm     = ('PRECTOTCORR', 'mean')
).round(2)

resumen_empresa.columns = [
    'Registros', 'Ventas Total (M kWh)', 'Ventas Prom (M kWh)',
    'Ingreso Total (B �)', 'Precio Medio (�/kWh)',
    'Temp Prom (�C)', 'Precipitacion Prom (mm)'
]
resumen_empresa

## 13. Analisis geografico por provincia y centrales

La vista  expone la ubicacion geografica de cada central electrica asociada a las empresas distribuidoras, incluyendo provincia, canton y distrito. Esto permite analizar la distribucion territorial del sistema electrico nacional y entender como se distribuye la infraestructura de generacion por region.

A continuacion cargamos esta informacion directamente desde la base de datos usando  del proyecto, replicando la misma vista SQL.

In [ ]:
pip install psycopg2-binary

In [ ]:
from datos.CargadorDatos import CargadorDatos

db = CargadorDatos()
db.GestorDB.port = 5433
db.GestorDB.user = "sa"
db.GestorDB.password = "progra"

In [ ]:
df_geo = db.GestorDB.consultar('''
    SELECT DISTINCT
        nombre_empresa,
        provincia,
        canton,
        central_electrica,
        fuente,
        operador,
        coordenada_x,
        coordenada_y
    FROM "Fact_Dim".prediccion_precio_mes
    WHERE provincia IS NOT NULL
    ORDER BY nombre_empresa, provincia
''')

print(f'Registros geograficos: {df_geo.shape[0]}')
df_geo.head()

### Centrales electricas por provincia

In [ ]:
# Cantidad de centrales por provincia
centrales_prov = df_geo.groupby('provincia')['central_electrica'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(centrales_prov.index, centrales_prov.values,
              color=PALETA[1], edgecolor='white', alpha=0.85)
for bar, val in zip(bars, centrales_prov.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', fontsize=9)
ax.set_title('Cantidad de centrales electricas por provincia')
ax.set_ylabel('Cantidad de centrales')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### Fuentes electricas por empresa

In [ ]:
fuentes = df_geo.groupby(['nombre_empresa', 'fuente'])['central_electrica'].nunique().reset_index()
fuentes.columns = ['Empresa', 'Fuente', 'Cantidad']

pivot_fuentes = fuentes.pivot_table(index='Empresa', columns='Fuente',
                                    values='Cantidad', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
pivot_fuentes.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white')
ax.set_title('Fuentes electricas por empresa (cantidad de centrales)')
ax.set_ylabel('Cantidad de centrales')
ax.set_xlabel('')
ax.legend(title='Fuente', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### Interpretacion

La distribucion geografica de las centrales electricas refleja la organizacion territorial del sistema electrico costarricense. Las provincias con mayor cantidad de centrales concentran tambien mayor capacidad de generacion, lo que se corresponde con las empresas de mayor volumen de ventas. En cuanto a las fuentes, Costa Rica mantiene una matriz predominantemente hidroelectrica, aunque algunas empresas integran fuentes termicas o renovables no convencionales en sus zonas de concesion. Esta informacion es relevante para el modelo predictivo ya que la fuente de generacion puede influir en la disponibilidad y precio del kWh segun condiciones hidrologicas o estacionales.

## 14. Analisis por bloque de consumo

La vista  incorpora la dimension de **bloque de consumo**, que desagrega cada tarifa en rangos especificos de kWh consumidos (por ejemplo: primeros 50 kWh, 51-150 kWh, mas de 150 kWh). Esta informacion permite entender como se distribuye el consumo real de los usuarios dentro de cada categoria tarifaria y cuanto pagan en promedio por bloque.

In [ ]:
df_bloque = db.consultar('''
    SELECT
        nombre_empresa,
        tarifa,
        bloque_consumo,
        tipo_tarifa,
        AVG(tarifa_promedio)   AS tarifa_promedio_avg,
        SUM(ventas_por_mes)    AS ventas_total,
        SUM(ingreso_con_cvg)   AS ingreso_total
    FROM Fact_Dim.vw_ingresos_empresas_tarifas_mensual
    WHERE bloque_consumo NOT IN ('SIN BLOQUE ASOCIADO')
      AND tarifa_promedio > 0
    GROUP BY nombre_empresa, tarifa, bloque_consumo, tipo_tarifa
    ORDER BY tarifa, bloque_consumo
''')

print(f'Registros: {df_bloque.shape[0]}')
df_bloque.head(10)

### Tarifa promedio por bloque de consumo

In [ ]:
bloque_avg = df_bloque.groupby('bloque_consumo')['tarifa_promedio_avg'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.barh(bloque_avg.index, bloque_avg.values,
               color=PALETA[2], edgecolor='white', alpha=0.85)
for bar, val in zip(bars, bloque_avg.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=8)
ax.set_title('Tarifa promedio por bloque de consumo (colones por kWh)')
ax.set_xlabel('Tarifa promedio (colones/kWh)')
plt.tight_layout()
plt.show()

### Ventas totales por bloque y empresa

In [ ]:
pivot_bloque = df_bloque.pivot_table(
    index='nombre_empresa', columns='bloque_consumo',
    values='ventas_total', aggfunc='sum', fill_value=0
)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot_bloque / 1e6, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Millones kWh'})
ax.set_title('Ventas por empresa y bloque de consumo (millones kWh)')
ax.set_ylabel('')
ax.set_xlabel('')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

### Interpretacion

El analisis por bloque de consumo revela que los bloques de mayor rango de kWh concentran la mayor parte del ingreso total, aunque representan una proporcion menor de usuarios. Esto confirma el principio tarifario progresivo que aplica ARESEP en Costa Rica: a mayor consumo, mayor tarifa por kWh. Los bloques bajos, asociados a consumidores de menor ingreso economico, tienen tarifas subsidiadas que se compensan con los bloques superiores. Para el modelo predictivo, el bloque de consumo es una variable descriptiva importante que complementa la tarifa y permite segmentar el comportamiento de consumo con mayor precision.

## 15. Analisis por fuente electrica y centrales asociadas

La vista  consolida la relacion entre cada empresa distribuidora y sus centrales electricas asociadas, incluyendo la cantidad de centrales, los operadores y las fuentes de generacion utilizadas. Este analisis permite entender la estructura de infraestructura detras de cada empresa y como se relaciona con su capacidad de distribucion.

In [ ]:
df_centrales = db.consultar('''
    SELECT
        empresa_canonica,
        cantidad_centrales_asociadas,
        fuentes_electricas_agregadas,
        centrales_electricas_agregadas,
        operadores_centrales_agregados
    FROM Fact_Dim.vw_empresa_centrales_agregadas
    ORDER BY cantidad_centrales_asociadas DESC
''')

df_centrales

### Centrales asociadas por empresa

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(
    df_centrales['empresa_canonica'],
    df_centrales['cantidad_centrales_asociadas'],
    color=PALETA[:len(df_centrales)], edgecolor='white'
)
for bar, val in zip(bars, df_centrales['cantidad_centrales_asociadas']):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            str(int(val)), va='center', fontsize=9)
ax.set_title('Cantidad de centrales electricas asociadas por empresa')
ax.set_xlabel('Cantidad de centrales')
plt.tight_layout()
plt.show()

### Ventas vs cantidad de centrales asociadas

In [ ]:
ventas_x_empresa = df.groupby('Empresa')['Ventas'].sum().reset_index()
ventas_x_empresa.columns = ['empresa_canonica', 'Ventas']
scatter_df = ventas_x_empresa.merge(df_centrales[['empresa_canonica','cantidad_centrales_asociadas']],
                                     on='empresa_canonica', how='inner')

fig, ax = plt.subplots(figsize=(10, 5))
for i, row in scatter_df.iterrows():
    ax.scatter(row['cantidad_centrales_asociadas'], row['Ventas']/1e9,
               s=120, color=PALETA[i % len(PALETA)], zorder=3)
    ax.annotate(row['empresa_canonica'],
                (row['cantidad_centrales_asociadas'], row['Ventas']/1e9),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_title('Centrales asociadas vs Ventas totales por empresa')
ax.set_xlabel('Cantidad de centrales asociadas')
ax.set_ylabel('Ventas totales (B kWh)')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### Interpretacion

La relacion entre la cantidad de centrales asociadas y el volumen de ventas no es perfectamente lineal, lo que indica que el tamano de la empresa no depende unicamente de la cantidad de infraestructura sino tambien de la escala de su zona de concesion y la densidad poblacional que atiende. ICE, como empresa nacional, tiene asociadas la mayor cantidad de centrales distribuidas en todo el territorio, mientras que empresas cooperativas como COOPEALFARORUIZ operan con pocas centrales pero cubren zonas geograficas especificas y bien delimitadas. Esta informacion enriquece el modelo predictivo al incorporar el contexto de infraestructura de cada empresa como variable contextual.

## 14. Conclusiones del EDA

El analisis exploratorio realizado sobre el dataset final que integra datos de ARESEP y NASA POWER para el periodo 2020-2025 revela una serie de hallazgos importantes que orientan la construccion del modelo supervisado de prediccion de consumo electrico en Costa Rica.

En primer lugar, la distribucion del mercado electrico es marcadamente asimetrica. **ICE** concentra la mayor parte de las ventas e ingresos a nivel nacional, con cifras que superan ampliamente a todas las demas empresas distribuidoras combinadas. Le sigue **CNFL** como segundo actor relevante, mientras que las cooperativas regionales como COOPELESCA, JASEC y ESPH operan a escalas significativamente menores pero con patrones de consumo propios de sus zonas geograficas de concesion.

En cuanto a la estacionalidad, se observa un patron moderado pero consistente: los meses de enero a marzo, correspondientes a la epoca seca en Costa Rica, registran un consumo ligeramente superior al resto del ano. Este comportamiento se relaciona directamente con el uso de ventilacion y sistemas de climatizacion durante las temperaturas mas altas, especialmente en zonas costeras como Guanacaste. El heatmap por empresa y mes confirma que esta estacionalidad no es uniforme entre distribuidoras, siendo COOPEGUANACASTE la que presenta los picos de verano mas pronunciados.

Las variables climaticas muestran una relacion estadisticamente significativa con el consumo electrico. La temperatura media mensual (T2M) y la temperatura maxima (T2M_MAX) presentan correlacion positiva con las ventas, lo que valida la hipotesis de que el calor impulsa el consumo. Por el contrario, la humedad relativa (RH2M) se correlaciona negativamente, asociandose las epocas lluviosas con menor demanda electrica. Estos hallazgos respaldan la inclusion de variables climaticas de NASA POWER como entradas del modelo de regresion.

El analisis tarifario evidencia que la **tarifa Residencial** lidera el volumen de ventas, seguida por la Industrial y Comercios y Servicios. Sin embargo, en terminos de precio por kWh, las tarifas especiales como Alumbrado Publico y Media Tension presentan estructuras diferenciadas. A lo largo del periodo analizado, todos los segmentos muestran una tendencia alcista en el precio medio, reflejo de los ajustes tarifarios aprobados periodicamente por ARESEP. El analisis por bloque de consumo complementa este panorama al mostrar que los usuarios de mayor consumo pagan tarifas progresivamente mas altas, en linea con la politica de subsidio cruzado vigente en el pais.

Desde el punto de vista geografico, la infraestructura de generacion se concentra principalmente en las provincias con mayor densidad hidrografica, dado el predominio de la fuente hidroelectrica en la matriz energetica costarricense. Las empresas con mayor cantidad de centrales asociadas no necesariamente son las de mayor volumen de ventas, lo que refleja la importancia de la densidad poblacional y el tipo de zona de concesion en la determinacion del consumo total.

Finalmente, los valores faltantes detectados en las variables climaticas corresponden exclusivamente a la empresa USUARIOS DIRECTOS, que carece de coordenadas geograficas en el sistema, por lo que no recibio datos de NASA POWER. Esta limitacion debe considerarse en la etapa de preprocesamiento del modelo. Con base en todos estos hallazgos, las variables recomendadas para el modelo supervisado son: mes, anio, empresa, tarifa, sistema, temperatura media, velocidad del viento, humedad relativa y precipitacion, con ventas en kWh como variable objetivo.